# Sprint 5: Late-fusion policy engine

Proposal weeks 9–10. The Attention U-Net and GraphSAGE stay separate. This notebook applies the policy matrix in `src/models/fusion.py` to their probability outputs, one action per parcel, and checks whether flood-responsive actions concentrate inside the NRRP high-risk zones.

| Flood hazard \ deprivation | low | medium | high |
|---|---|---|---|
| low (P < 0.5) | retain and formalise | retain and upgrade | priority in-situ upgrade |
| high (P ≥ 0.5, or inside an NRRP zone) | flood-resilient upgrade | flood-resilient upgrade | compensated resettlement |

Compensated resettlement also needs a majority high-deprivation probability (`P(high) > 0.5`). A plurality is rewritten to in-place resilient upgrade. The NRRP zone forces the high-hazard row. It does not skip that deprivation gate.

The Sprint 5 exit bar is the same lift statistic as the Sprint 2 flood label: model flood-responsive actions, scored before the NRRP override, must concentrate inside the reference (`lift > config.FUSION_NRRP_LIFT_TARGET`). The override is kept out of that score so zone membership cannot manufacture the lift. A justice audit checks the displacement rule on every run.

**Offline / Cloud Agent note.** Section 2 runs on synthetic parcels and is the offline check. Section 3 needs Drive checkpoints, parcel coordinates (`data.pos`, written by a Sprint 5 graph rebuild), and either a flood-probability GeoTIFF or the stack + U-Net checkpoint + tiles. Real lift is a Colab number. Do not quote section 2 as a settlement result.


## 0. Before you run

- **Runtime → Change runtime type → T4 GPU** is optional. The matrix and GraphSAGE inference run on CPU. U-Net tile inference is faster on a GPU.
- Mount Drive so graphs and checkpoints persist.
- Sprint 4 graphs are enough for deprivation probabilities. Graphs saved before Sprint 5 have no building centroids. Section 3 fetches OSM once and attaches them when the node features still match. `REBUILD_GRAPHS = True` forces that fetch even when centroids are already present.
- Set `NRRP_GEOJSON_PATH` to the 37 high-risk zone polygons when you have them. With it unset, the notebook still assigns actions and skips the lift bar.
- Flood input is a per-settlement probability GeoTIFF if you have one. Otherwise the notebook runs the Sprint 4 U-Net on un-augmented tiles and georeferences them with the Drive export `{settlement}_raster_stack.tif` (Earth Engine's ` (1).tif` copies included).


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — GraphSAGE and the policy matrix run on CPU.")


In [ ]:
%%bash
# Prefer the Sprint 5 PR branch while open; switch BRANCH to main after merge.
BRANCH=cursor/sprint5-late-fusion-d3fc
REPO=https://github.com/emmanuelamor5/nairobi-risk-mapping.git

if [ -d /content/nairobi-risk-mapping/.git ]; then
  cd /content/nairobi-risk-mapping
  git fetch origin "$BRANCH" || git fetch origin main
  git checkout "$BRANCH" 2>/dev/null || git checkout main
  git pull --ff-only origin "$BRANCH" 2>/dev/null || git pull --ff-only origin main
else
  git clone "$REPO" /content/nairobi-risk-mapping
  cd /content/nairobi-risk-mapping
  git fetch origin "$BRANCH" || true
  git checkout "$BRANCH" 2>/dev/null || git checkout main
fi
cd /content/nairobi-risk-mapping
git rev-parse --abbrev-ref HEAD
ls src/models/fusion.py


In [ ]:
%cd /content/nairobi-risk-mapping
!pip install -q -r requirements.txt


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/nairobi_risk_mapping")
except ImportError:
    print("Not in Colab — artefacts stay under the repo working directory.")
    DRIVE_ROOT = Path(".")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

DATA_LINK = Path("data")
if not (DATA_LINK.is_symlink() or DATA_LINK.exists()):
    target = DRIVE_ROOT / "data"
    target.mkdir(parents=True, exist_ok=True)
    DATA_LINK.symlink_to(target)

SETTLEMENT_KEYS = ("kibera", "mathare", "mukuru")
REBUILD_GRAPHS = False
# 37 NRRP high-risk zone polygons. None skips the lift bar (actions still run).
NRRP_GEOJSON_PATH = None
# Optional explicit flood-probability GeoTIFFs, keyed by settlement.
FLOOD_RASTER_PATHS = {}
# Optional stack GeoTIFFs used only to georeference U-Net tiles.
STACK_PATHS = {}
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "sprint5"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA ->", DATA_LINK.resolve())
print("CHECKPOINTS ->", CHECKPOINT_DIR)


## 1. Policy matrix

The matrix is code, not a comment. This cell prints the same table `decide_policy` applies.


In [ ]:
from src import config
from src.models.fusion import (
    POLICY_ACTIONS,
    action_labels,
    decide_policy,
    describe_policy_matrix,
    format_policy_report,
    justice_audit,
    summarize_policy,
    validate_against_reference,
)

assert config.FUSION_FLOOD_THRESHOLD == 0.5
assert config.FUSION_NRRP_LIFT_TARGET == 1.0
assert POLICY_ACTIONS[-1] == "compensated_resettlement"
print(describe_policy_matrix())
print("lift target: strictly greater than", config.FUSION_NRRP_LIFT_TARGET)


## 2. Synthetic worked example

Six parcels, one per matrix cell, then a plurality case the majority gate must rewrite, then 100 parcels whose high-hazard predictions sit inside a reference mask. The lift in this section is a check that the statistic moves when concentration is built in. It is not a Kibera, Mathare, or Mukuru result.


In [ ]:
import numpy as np

def _one_hot(cls, n=1):
    dep = np.zeros((n, 3), dtype=np.float64)
    dep[:, cls] = 1.0
    return dep

# One parcel per matrix cell: flood in {0.1, 0.9} x class in {0, 1, 2}.
flood_cells = np.array([0.1, 0.1, 0.1, 0.9, 0.9, 0.9])
dep_cells = np.vstack([_one_hot(c) for c in (0, 1, 2, 0, 1, 2)])
cells = decide_policy(flood_cells, dep_cells)
print("matrix cells:")
labels = action_labels(cells["action_index"])
for i, label in enumerate(labels):
    print(f"  flood={flood_cells[i]:.1f}  class={int(cells['deprivation_class'][i])}  -> {label}")

# Plurality high-deprivation on a wet parcel must not be displaced.
plurality = decide_policy([0.9], np.array([[0.40, 0.15, 0.45]]))
print("plurality gate fired:", bool(plurality["majority_gate_applied"][0]),
      "displaced:", bool(plurality["displacement"][0]))

# Constructed concentration: the first 20 of 100 parcels are high hazard and inside the reference.
n = 100
flood = np.zeros(n)
flood[:20] = 0.95
inside = np.zeros(n, dtype=bool)
inside[:20] = True
synthetic = decide_policy(flood, _one_hot(0, n), inside_nrrp=inside)
synthetic_validation = validate_against_reference(synthetic, inside)
synthetic_justice = justice_audit(synthetic)
print()
print("SYNTHETIC — not a settlement result")
print(format_policy_report(
    summarize_policy(synthetic), synthetic_validation, synthetic_justice,
))
assert synthetic_justice["passes"]
assert synthetic_validation["meets_lift_target"]


## 3. Real heads, when Drive artefacts exist

Deprivation probabilities come from a saved softmax (Sprint 3/4) or from `GraphSAGEModel.predict_proba` on the settlement graph. Flood probabilities come from a GeoTIFF when one is present, otherwise from the U-Net on un-augmented tiles. Settlements missing either head are skipped. The lift is computed only when every fused settlement has a reference mask.


In [ ]:
import json

import numpy as np
import torch

from src.data_pipeline import DataPipeline
from src.models.attention_unet_model import AttentionUNetModel
from src.models.dataset import DEFAULT_LAYOUT, EVENT_S1_LAYOUT
from src.models.fusion import (
    aggregate_tile_probabilities,
    newest_geotiff,
    parcel_reference_mask,
    parcel_rowcol,
    sample_geotiff_at_pos,
    select_orig_manifest_entries,
)
from src.models.graphsage_model import GraphSAGEModel
from src.models.multi_settlement import load_settlement_graphs
from src.utils.graph_utils import align_parcel_positions, graph_has_positions

if REBUILD_GRAPHS:
    for key in SETTLEMENT_KEYS:
        print("Rebuilding graph for", key)
        DataPipeline(settlement_key=key).acquire_graph_data()

graphs = load_settlement_graphs(Path("data/graphs"), SETTLEMENT_KEYS)
print("graphs:", {k: int(g.num_nodes) for k, g in graphs.items()} or "none")


def _torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def _deprivation_probs(key, graph):
    candidates = [
        DRIVE_ROOT / "checkpoints" / "sprint4" / "graphsage" / f"{key}_deprivation_probs.pt",
        DRIVE_ROOT / "checkpoints" / "sprint3_graphsage" / f"{key}_deprivation_probs.pt",
        Path("checkpoints/sprint3_graphsage") / f"{key}_deprivation_probs.pt",
        Path("checkpoints/sprint4/graphsage") / f"{key}_deprivation_probs.pt",
    ]
    for path in candidates:
        if not path.exists():
            continue
        blob = _torch_load(path)
        probs = blob["probs"] if isinstance(blob, dict) else blob
        if torch.is_tensor(probs):
            probs = probs.detach().cpu().numpy()
        probs = np.asarray(probs, dtype=np.float64)
        if probs.shape[0] == graph.num_nodes and probs.shape[1] == 3:
            print(f"  {key}: deprivation from {path}")
            return probs
        print(f"  {key}: {path} shape {probs.shape} != graph {(graph.num_nodes, 3)}")

    ckpts = [
        DRIVE_ROOT / "checkpoints" / "sprint4" / "graphsage" / "graphsage_best.pt",
        Path("checkpoints/sprint4/graphsage/graphsage_best.pt"),
        DRIVE_ROOT / "checkpoints" / "sprint3_graphsage" / "graphsage_best.pt",
        Path("checkpoints/sprint3_graphsage/graphsage_best.pt"),
    ]
    ckpt = next((p for p in ckpts if p.exists()), None)
    if ckpt is None:
        print(f"  {key}: no deprivation checkpoint")
        return None
    model = GraphSAGEModel(
        in_channels=int(graph.x.size(1)),
        hidden_channels=64,
        num_classes=config.DEPRIVATION_NUM_CLASSES,
        dropout=0.5,
    )
    try:
        model.load_checkpoint(str(ckpt))
    except Exception as exc:
        print(f"  {key}: failed to load {ckpt}: {exc}")
        return None
    probs = model.predict_proba(graph).numpy()
    print(f"  {key}: deprivation from predict_proba ({ckpt})")
    return np.asarray(probs, dtype=np.float64)


def _search_dirs():
    return [DRIVE_ROOT, DRIVE_ROOT / "data" / "raster_stacks", Path("data/raster_stacks")]


def _ensure_positions(key, graph):
    """Attach centroids to a pre-Sprint-5 graph. One OSM fetch, then saved."""
    if graph_has_positions(graph):
        return graph
    src_path = Path("data/graphs") / f"{key}_graph.pt"
    backup = Path("data/graphs") / f"{key}_graph_before_pos.pt"
    if src_path.exists() and not backup.exists():
        backup.write_bytes(src_path.read_bytes())
        print(f"  {key}: backed up the graph without centroids to {backup.name}")
    print(f"  {key}: no centroids on the saved graph. Fetching OSM once to attach them.")
    print("    GraphSAGE is not retrained. This is the slow step (Overpass).")
    refreshed, _ = DataPipeline(settlement_key=key).acquire_graph_data()
    updated, status = align_parcel_positions(graph, refreshed)
    if status == "aligned":
        torch.save(updated, src_path)
        print(f"  {key}: centroids aligned onto the existing {updated.num_nodes} nodes.")
    else:
        print(
            f"  {key}: OSM node set changed ({graph.num_nodes} -> {refreshed.num_nodes}). "
            f"Using the fresh graph. Previous file kept as {backup.name}."
        )
        updated = refreshed
    return updated


def _flood_raster(key, graph):
    if not graph_has_positions(graph):
        print(f"  {key}: still no centroids after the OSM attach")
        return None
    raster = None
    if key in FLOOD_RASTER_PATHS and FLOOD_RASTER_PATHS[key]:
        candidate = Path(FLOOD_RASTER_PATHS[key])
        if candidate.exists():
            raster = candidate
    if raster is None:
        raster = newest_geotiff(_search_dirs(), f"{key}_flood_proba")
    if raster is None:
        return None
    pos_crs = getattr(graph, "pos_crs", config.CRS_METRIC)
    sampled = sample_geotiff_at_pos(raster, graph.pos.detach().cpu().numpy(), pos_crs)
    finite = int(np.isfinite(sampled).sum())
    print(f"  {key}: flood raster {raster}  finite {finite}/{sampled.size}")
    return sampled


def _tile_array(entry, tiles_dir):
    path = Path(entry["path"])
    if not path.exists():
        path = tiles_dir / f"{entry['tile_id']}.npy"
    if not path.exists():
        return None
    return np.load(path)


def _flood_from_unet(key, graph):
    if not graph_has_positions(graph):
        return None
    stack = None
    if key in STACK_PATHS and STACK_PATHS[key]:
        candidate = Path(STACK_PATHS[key])
        if candidate.exists():
            stack = candidate
    if stack is None:
        # Sprint 1–4 exports land in the Drive folder as
        # {key}_raster_stack.tif, not data/raster_stacks/{key}_stack.tif.
        stack = newest_geotiff(_search_dirs(), f"{key}_raster_stack")
    tiles_dir = Path("data/tiles") / key
    if not tiles_dir.exists():
        tiles_dir = DRIVE_ROOT / "data" / "tiles" / key
    ckpts = [
        DRIVE_ROOT / "checkpoints" / "sprint4" / "attention_unet" / "attention_unet_best.pt",
        Path("checkpoints/sprint4/attention_unet/attention_unet_best.pt"),
        DRIVE_ROOT / "checkpoints_v2" / "attention_unet_best.pt",
        Path("checkpoints_v2/attention_unet_best.pt"),
        DRIVE_ROOT / "checkpoints" / "attention_unet_best.pt",
    ]
    ckpt = next((p for p in ckpts if p.exists()), None)
    if stack is None or ckpt is None or not tiles_dir.exists():
        print(f"  {key}: no flood raster, and U-Net inference needs a stack GeoTIFF, "
              f"orig tiles, and a checkpoint (stack={stack is not None}, "
              f"tiles={tiles_dir.exists()}, ckpt={ckpt is not None})")
        return None

    entries = []
    for split in ("train", "val", "test"):
        manifest = tiles_dir / f"manifest_{split}.json"
        if manifest.exists():
            entries.extend(json.loads(manifest.read_text()))
    windows = select_orig_manifest_entries(entries)
    if not windows:
        print(f"  {key}: manifests have no un-augmented tiles")
        return None

    sample = _tile_array(windows[0], tiles_dir)
    if sample is None:
        print(f"  {key}: tile file missing for {windows[0].get('tile_id')} "
              f"(tried {windows[0].get('path')} and {tiles_dir})")
        return None
    n_bands = int(sample.shape[0])
    if n_bands == len(DEFAULT_LAYOUT.band_names):
        layout = DEFAULT_LAYOUT
    elif n_bands == len(EVENT_S1_LAYOUT.band_names):
        layout = EVENT_S1_LAYOUT
    else:
        print(f"  {key}: tile has {n_bands} bands; expected "
              f"{len(DEFAULT_LAYOUT.band_names)} or {len(EVENT_S1_LAYOUT.band_names)}")
        return None

    model = AttentionUNetModel(layout=layout, dropout=0.0)
    try:
        model.load_checkpoint(str(ckpt))
    except Exception as exc:
        print(f"  {key}: failed to load U-Net {ckpt}: {exc}")
        return None

    tile_probs = []
    for entry in windows:
        tile = _tile_array(entry, tiles_dir)
        if tile is None:
            print(f"  {key}: skipping missing tile {entry.get('tile_id')}")
            continue
        inputs = np.nan_to_num(tile[layout.input_indices], nan=0.0)
        batch = torch.from_numpy(np.ascontiguousarray(inputs)).float().unsqueeze(0)
        proba = model.predict_proba(batch).numpy()[0, 0]
        tile_probs.append({
            "row_off": int(entry["row_off"]),
            "col_off": int(entry["col_off"]),
            "proba": proba,
        })
    if not tile_probs:
        print(f"  {key}: every orig tile file was missing")
        return None

    import rasterio
    with rasterio.open(stack) as src:
        transform = src.transform
        raster_crs = src.crs
    rows, cols = parcel_rowcol(
        graph.pos.detach().cpu().numpy(),
        transform,
        pos_crs=getattr(graph, "pos_crs", config.CRS_METRIC),
        raster_crs=raster_crs,
    )
    sampled = aggregate_tile_probabilities(rows, cols, tile_probs)
    finite = int(np.isfinite(sampled).sum())
    print(f"  {key}: flood from U-Net tiles ({ckpt})  finite {finite}/{sampled.size}")
    return sampled


def _parcel_frame(graph):
    import geopandas as gpd
    from shapely.geometry import Point

    pos = graph.pos.detach().cpu().numpy()
    crs = getattr(graph, "pos_crs", config.CRS_METRIC)
    return gpd.GeoDataFrame(
        geometry=[Point(float(x), float(y)) for x, y in pos],
        crs=crs,
    )


def _load_reference():
    """NRRP polygons when a path is set; otherwise one OSM river-corridor fetch."""
    import geopandas as gpd
    from src.utils.graph_utils import fetch_river_corridor

    if NRRP_GEOJSON_PATH and Path(NRRP_GEOJSON_PATH).exists():
        print("Lift reference: NRRP polygons", NRRP_GEOJSON_PATH)
        return gpd.read_file(NRRP_GEOJSON_PATH), "NRRP zones"
    print(
        f"NRRP_GEOJSON_PATH is unset. Lift uses an OSM river/stream/drain "
        f"corridor ({config.FUSION_RIVER_BUFFER_M:.0f} m), one Overpass query. "
        f"That is a coarser prior than the 37 NRRP zones."
    )
    try:
        corridor = fetch_river_corridor(SETTLEMENT_KEYS, buffer_m=config.FUSION_RIVER_BUFFER_M)
    except Exception as exc:
        print("River-corridor fetch failed:", exc)
        print("Lift will be skipped. Re-run later, or set NRRP_GEOJSON_PATH.")
        return None, None
    print(f"River corridor polygons: {len(corridor)}")
    if len(corridor) == 0:
        print("No river, stream, or drain lines came back. Lift will be skipped.")
        return None, None
    return corridor, "OSM river corridor"


def _reference_mask(key, graph, zones, source):
    if zones is None or len(zones) == 0:
        return None
    if not graph_has_positions(graph):
        print(f"  {key}: cannot test {source} membership without centroids")
        return None
    mask = parcel_reference_mask(_parcel_frame(graph), zones)
    print(f"  {key}: {int(mask.sum())}/{mask.size} parcels inside the {source}")
    return mask


flood_parts, dep_parts, ref_parts, sid_parts = [], [], [], []
keys_used = []
reference_zones, reference_source = _load_reference()
for key in SETTLEMENT_KEYS:
    graph = graphs.get(key)
    if graph is None:
        print(f"  {key}: graph missing")
        continue
    print(key)
    try:
        graph = _ensure_positions(key, graph)
    except Exception as exc:
        print(f"  {key}: could not attach centroids ({exc})")
        continue
    graphs[key] = graph
    dep = _deprivation_probs(key, graph)
    flood = _flood_raster(key, graph)
    if flood is None:
        flood = _flood_from_unet(key, graph)
    if dep is None or flood is None:
        print(f"  {key}: skipped — both heads are required")
        continue
    if dep.shape[0] != flood.shape[0]:
        print(f"  {key}: length mismatch dep {dep.shape[0]} flood {flood.shape[0]}")
        continue
    ref = _reference_mask(key, graph, reference_zones, reference_source)
    flood_parts.append(flood)
    dep_parts.append(dep)
    ref_parts.append(ref)
    sid_parts.append(np.full(flood.shape[0], len(keys_used), dtype=np.int64))
    keys_used.append(key)

real_decision = None
real_summary = None
real_validation = None
real_justice = None
settlement_index = None

if not keys_used:
    print("No settlement had both heads. Section 2 exercised the engine on synthetic parcels.")
    print("Real actions need graphs + a GraphSAGE checkpoint + a flood raster or U-Net tiles.")
else:
    inside = None
    if all(part is not None for part in ref_parts):
        inside = np.concatenate(ref_parts)
    elif any(part is not None for part in ref_parts):
        print("Reference mask is missing for at least one fused settlement — lift will be skipped.")
    # The official zones are a planning constraint and force the high-hazard
    # row. The river corridor is only the lift reference: treating every
    # drain buffer as an NRRP zone would resettle buildings the U-Net calls dry.
    override = inside if reference_source == "NRRP zones" else None
    real_decision = decide_policy(
        np.concatenate(flood_parts),
        np.concatenate(dep_parts),
        inside_nrrp=override,
    )
    settlement_index = np.concatenate(sid_parts)
    real_summary = summarize_policy(
        real_decision, settlement_index=settlement_index, settlement_keys=keys_used,
    )
    real_justice = justice_audit(real_decision)
    real_validation = None
    if inside is not None:
        real_validation = validate_against_reference(real_decision, inside)
    print()
    print("REAL HEADS —", ", ".join(keys_used))
    print(format_policy_report(real_summary, real_validation, real_justice))
    if real_validation is None:
        print("Lift not measured. The river-corridor fetch failed and NRRP_GEOJSON_PATH is unset.")
    elif reference_source != "NRRP zones":
        print(
            "Lift is against the OSM river corridor, not the 37 NRRP zones. "
            "Set NRRP_GEOJSON_PATH to score the official high-risk map."
        )
    if not real_justice["passes"]:
        raise RuntimeError("Justice audit failed — resettlement escaped the majority-hazard rule.")


## 4. Save the parcel table

Sprint 6 reads this file for the dashboard. Action names are not stored; `action_labels` maps `action_index` back to the five policy names (`-1` is insufficient evidence).


In [ ]:
from src.models.fusion import save_policy_table

if real_decision is None:
    print("Nothing to save. The synthetic section did not write a parcel table.")
else:
    out = CHECKPOINT_DIR / "policy_table.npz"
    save_policy_table(
        out,
        real_decision,
        settlement_index=settlement_index,
        settlement_keys=keys_used,
    )
    print("Saved", out, "parcels", real_decision["n"])


## Done

- The policy matrix, the majority gate on resettlement, and the non-circular lift bar are in `src/models/fusion.py` and covered by `tests/test_fusion.py`.
- Quote a Sprint 5 lift only from section 3, after it prints `meets=True` or `meets=False` on real heads and the NRRP polygons (or a river-corridor reference you pass as `NRRP_GEOJSON_PATH`).
- Next is Sprint 6: SHAP on GraphSAGE, Grad-CAM on the attention gates, and the Folium dashboard. Do not start that here.
